# Importaciones necesarias

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Carga de datos
Tras las importaciones procedemos a cargar los datos. Aquellas variables con valor 0 serán reemplazados por la mediana. 

In [ ]:
df = pd.read_csv('/workspace/diabetes.csv')

cols_with_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in cols_with_zeros:
    df[col] = df[col].replace(0, df[col].median())

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,30.5,33.6,0.627,50,1
1,1,85,66,29,30.5,26.6,0.351,31,0
2,8,183,64,23,30.5,23.3,0.672,32,1
3,1,89,66,23,94.0,28.1,0.167,21,0
4,0,137,40,35,168.0,43.1,2.288,33,1


# Preparar el dataset para la clasificación
Separaremos las variables del objetivo y preparamos los datos. 

In [3]:
X = df.drop(columns=['Outcome']).values
y = df['Outcome'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1) 

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# Construcción de la red neuronal
La red estará formada por una capa de entrada, dos ocultas (64 y 32 neuronas) y una capa de salida con una neurona y la función de activación sigmoide. 

In [4]:
class DiabetesClassifier(nn.Module):
    def __init__(self, input_dim):
        super(DiabetesClassifier, self).__init__()
        self.layer1 = nn.Linear(input_dim, 64)
        self.relu1 = nn.ReLU()
        self.layer2 = nn.Linear(64, 32)
        self.relu2 = nn.ReLU()
        self.output_layer = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.layer1(x)
        out = self.relu1(out)
        out = self.layer2(out)
        out = self.relu2(out)
        out = self.output_layer(out)
        out = self.sigmoid(out) # Devuelve probabilidad entre 0 y 1
        return out

input_size = X_train_tensor.shape[1]
model = DiabetesClassifier(input_size)

# Entrenamiento del modelo
Configuramos la pérdida binaria y el optimizador Adam. Se entrenará el modelo a través de 100 épocas. 

In [5]:
criterion = nn.BCELoss() # Binary Cross-Entropy Loss
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 100

for epoch in range(epochs):
    model.train() 
    
    outputs = model(X_train_tensor)
    
    loss = criterion(outputs, y_train_tensor)
    
    optimizer.zero_grad() 
    loss.backward()       
    optimizer.step()      
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}')

Epoch [10/100], Loss: 0.6674
Epoch [20/100], Loss: 0.6197
Epoch [30/100], Loss: 0.5586
Epoch [40/100], Loss: 0.5021
Epoch [50/100], Loss: 0.4644
Epoch [60/100], Loss: 0.4420
Epoch [70/100], Loss: 0.4259
Epoch [80/100], Loss: 0.4150
Epoch [90/100], Loss: 0.4071
Epoch [100/100], Loss: 0.4000


# Evaluación del modelo
Por último, procedemos a evaluar el modelo. 

In [6]:
model.eval()

with torch.no_grad():
    test_predictions = model(X_test_tensor)
    
    test_predictions_class = (test_predictions >= 0.5).float()

y_true = y_test_tensor.numpy()
y_pred = test_predictions_class.numpy()

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"Accuracy general: {acc:.4f}")
print(f"F1 Score: {f1:.4f}\n")
print("--- Classification Report ---")
print(classification_report(y_true, y_pred))

Accuracy general: 0.7500
F1 Score: 0.6471

--- Classification Report ---
              precision    recall  f1-score   support

         0.0       0.80      0.81      0.81       123
         1.0       0.66      0.64      0.65        69

    accuracy                           0.75       192
   macro avg       0.73      0.73      0.73       192
weighted avg       0.75      0.75      0.75       192

